# Eksplorasi, Perbandingan, dan Analisis 6 Arsitektur CNN pada CIFAR-10
**Mata Kuliah:** Deep Learning (IF25-40401) — Institut Teknologi Sumatera
**Pertemuan 5:** CNN dan Variasinya

| | |
|---|---|
| **Nama** | ............................................ |
| **NIM** | ............................................ |
| **Kelas** | ............................................ |

### Spesifikasi eksperimen (sesuai ketentuan tugas)
| Ketentuan | Nilai |
|---|---|
| Dataset | CIFAR-10, pembagian **default torchvision**: `train=True` (50.000 citra) dan `train=False` (10.000 citra) |
| Random seed | **42** |
| Batch size | **64** |
| Optimizer | **Adam**, learning rate **0,001** |
| Epoch | **10** |
| Metrik evaluasi | **Akurasi** pada data uji |
| Model | LeNet-5, AlexNet, VGG-16, GoogLeNet, **ResNet-18**, **MobileNetV2** (2 model pilihan) |

### Keputusan desain
1. **Semua model dilatih dari nol (tanpa bobot pretrained) pada resolusi asli 32×32.** Karena hyperparameter dikunci sama untuk semua model, perbedaan akurasi hanya berasal dari **arsitektur**, bukan dari pretraining.
2. **Adaptasi ke citra 32×32.** AlexNet, VGG, GoogLeNet, ResNet, dan MobileNet dirancang untuk citra 224×224. Bagian awal (*stem*) yang memperkecil resolusi secara agresif disesuaikan, sedangkan **blok khas tiap arsitektur dipertahankan**. Cara ini juga dipakai paper ResNet untuk eksperimen CIFAR-10. Rincian adaptasi ada di Bagian 4.
3. **Tanpa augmentasi data** dan tanpa *learning-rate scheduler*, mengikuti notebook praktikum dosen, agar kondisi pelatihan sesederhana dan seadil mungkin.

> **Cara menjalankan:** Runtime → Change runtime type → **T4 GPU**, lalu **Runtime → Run all** (±35–50 menit). Setelah selesai, cetak ke PDF lewat **File → Print → Save as PDF** (lihat Bagian 11).

## 1. Konfigurasi dan Setup

In [ ]:
CONFIG = {
    "SEED": 42,
    "BATCH_SIZE": 64,
    "LR": 0.001,
    "EPOCHS": 10,
    "AMP": True,            # mixed precision di GPU: ±2x lebih cepat, hyperparameter tidak berubah
    "DIR_DATA": "./data",
    "DIR_HASIL": "hasil",   # tabel, grafik, dan riwayat training disimpan di sini
}
MODEL_DIPAKAI = ["LeNet-5", "AlexNet", "VGG-16", "GoogLeNet", "ResNet-18", "MobileNetV2"]

# Opsional: simpan hasil ke Google Drive agar tidak hilang bila runtime Colab ter-reset.
# from google.colab import drive; drive.mount("/content/drive")
# CONFIG["DIR_HASIL"] = "/content/drive/MyDrive/tugas_cnn_cifar10"

In [ ]:
import os, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets
from torch.utils.flop_counter import FlopCounterMode

def set_seed(seed):
    """Mengunci semua sumber keacakan: Python, NumPy, PyTorch (CPU & GPU), dan cuDNN."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = CONFIG["AMP"] and device.type == "cuda"
os.makedirs(CONFIG["DIR_HASIL"], exist_ok=True)

# Palet warna tetap per model (urutan tidak berubah di semua grafik) + bentuk penanda berbeda
WARNA = dict(zip(MODEL_DIPAKAI, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
PENANDA = dict(zip(MODEL_DIPAKAI, ["o", "s", "^", "D", "v", "P"]))
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

print("=" * 62)
print(f"PyTorch / torchvision : {torch.__version__} / {torchvision.__version__}")
print(f"Perangkat             : {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))
print(f"Mixed precision (AMP) : {USE_AMP}")
print("Konfigurasi           :", {k: v for k, v in CONFIG.items() if k in ("SEED", "BATCH_SIZE", "LR", "EPOCHS")})
print("=" * 62)
if device.type != "cuda":
    print("PERINGATAN: GPU tidak aktif. Di Colab: Runtime -> Change runtime type -> T4 GPU")

## 2. Eksplorasi Dataset CIFAR-10
Dataset dimuat dengan **pembagian default torchvision**: `train=True` untuk data latih dan `train=False` untuk data uji. Tidak ada data yang dipindahkan antar-split.

In [ ]:
train_set = datasets.CIFAR10(CONFIG["DIR_DATA"], train=True, download=True)
test_set = datasets.CIFAR10(CONFIG["DIR_DATA"], train=False, download=True)
KELAS = train_set.classes

print(f"Data latih : {len(train_set):,} citra | shape array {train_set.data.shape} | dtype {train_set.data.dtype}")
print(f"Data uji   : {len(test_set):,} citra | shape array {test_set.data.shape}")
print(f"Jumlah kelas: {len(KELAS)} -> {KELAS}")

jumlah_latih = np.bincount(train_set.targets, minlength=10)
jumlah_uji = np.bincount(test_set.targets, minlength=10)
print("\nDistribusi kelas (latih / uji):")
for k, a, b in zip(KELAS, jumlah_latih, jumlah_uji):
    print(f"  {k:<11}: {a:>5,} / {b:>5,}")
print("-> Dataset seimbang: setiap kelas punya jumlah citra yang sama, sehingga akurasi adalah metrik yang adil.")

In [ ]:
rng = np.random.default_rng(CONFIG["SEED"])
fig, axs = plt.subplots(2, 10, figsize=(15, 3.4))
for c in range(10):
    idx = rng.choice(np.where(np.array(train_set.targets) == c)[0], size=2, replace=False)
    for r in range(2):
        axs[r, c].imshow(train_set.data[idx[r]])
        axs[r, c].axis("off")
    axs[0, c].set_title(KELAS[c], fontsize=9)
plt.suptitle("Dua contoh citra per kelas (resolusi asli 32×32 piksel)", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["DIR_HASIL"], "contoh_data.png"), bbox_inches="tight")
plt.show()

### Normalisasi
Mean dan standar deviasi per kanal dihitung **hanya dari data latih**, lalu dipakai untuk menormalkan data latih dan data uji. Data uji tidak boleh dipakai untuk menghitung statistik apa pun. Hasilnya setara dengan `transforms.ToTensor()` + `transforms.Normalize(mean, std)`.

In [ ]:
def ke_tensor(ds):
    x = torch.from_numpy(ds.data).permute(0, 3, 1, 2).contiguous()   # (N, 32, 32, 3) -> (N, 3, 32, 32), uint8
    return x, torch.tensor(ds.targets)

x_train, y_train = ke_tensor(train_set)
x_test, y_test = ke_tensor(test_set)

MEAN = torch.stack([x_train[:, c].float().div(255).mean() for c in range(3)])
STD = torch.stack([x_train[:, c].float().div(255).std() for c in range(3)])
print("Mean per kanal (R, G, B):", MEAN.numpy().round(4))
print("Std  per kanal (R, G, B):", STD.numpy().round(4))

### DataLoader
Seluruh dataset (±180 MB dalam format `uint8`) disimpan di GPU, lalu dinormalisasi per batch. Cara ini jauh lebih cepat daripada `DataLoader` biasa di Colab yang CPU-nya hanya 2 core, dan hasilnya identik.
Urutan acak (shuffle) dikendalikan oleh `torch.Generator` dengan seed 42 yang dibuat ulang untuk setiap model, sehingga **keenam model melihat urutan batch yang persis sama**.

In [ ]:
class CIFARLoader:
    def __init__(self, x, y, batch_size, shuffle, seed=None):
        self.x, self.y = x.to(device), y.to(device)
        self.bs, self.shuffle = batch_size, shuffle
        self.gen = torch.Generator().manual_seed(seed) if shuffle else None
        self.mean = MEAN.view(1, 3, 1, 1).to(device)
        self.std = STD.view(1, 3, 1, 1).to(device)

    def __len__(self):
        return (len(self.x) + self.bs - 1) // self.bs

    def __iter__(self):
        n = len(self.x)
        idx = torch.randperm(n, generator=self.gen) if self.shuffle else torch.arange(n)
        idx = idx.to(device)
        for k in range(0, n, self.bs):
            b = idx[k:k + self.bs]
            x = (self.x[b].float() / 255.0 - self.mean) / self.std
            yield x, self.y[b]

def buat_loader(seed):
    train_loader = CIFARLoader(x_train, y_train, CONFIG["BATCH_SIZE"], shuffle=True, seed=seed)
    test_loader = CIFARLoader(x_test, y_test, 1000, shuffle=False)
    return train_loader, test_loader

_tl, _ = buat_loader(CONFIG["SEED"])
xb, yb = next(iter(_tl))
print(f"Satu batch: x {tuple(xb.shape)} {xb.dtype}, y {tuple(yb.shape)} | "
      f"mean {xb.mean():.3f}, std {xb.std():.3f} | {len(_tl)} iterasi per epoch")
del _tl, xb, yb

## 3. Alat Ukur Model
- **Parameter**: jumlah bobot yang dilatih (`p.numel()`).
- **MAC (multiply-accumulate)**: jumlah operasi kali-tambah untuk satu citra. `FlopCounterMode` menghitung FLOP = 2 × MAC, jadi hasilnya dibagi 2 (sama seperti notebook praktikum 04).
- **Porsi parameter di FC**: menguji pola "parameter menumpuk di fully connected" (slide 8 dan 14).

In [ ]:
def hitung_param(model):
    return sum(p.numel() for p in model.parameters())

def hitung_param_fc(model):
    return sum(p.numel() for m in model.modules() if isinstance(m, nn.Linear) for p in m.parameters())

@torch.no_grad()
def hitung_mac(model, input_size=(1, 3, 32, 32)):
    model.eval()
    x = torch.randn(*input_size, device=next(model.parameters()).device)
    with FlopCounterMode(display=False) as fc:
        model(x)
    return fc.get_total_flops() / 2

LEWATI = (nn.ReLU, nn.ReLU6, nn.Tanh, nn.BatchNorm2d, nn.Dropout, nn.Flatten, nn.Identity)

@torch.no_grad()
def jejak_shape(model, input_size=(1, 3, 32, 32)):
    """Mencatat shape output tiap layer/blok utama (aktivasi, BN, dropout dilewati)."""
    target = []
    for nama, anak in model.named_children():
        if isinstance(anak, nn.Sequential) and len(anak) > 1:
            target += [(f"{nama}.{n}", m) for n, m in anak.named_children() if not isinstance(m, LEWATI)]
        elif not isinstance(anak, LEWATI):
            target.append((nama, anak))
    log, hooks = [], []
    for nama, m in target:
        hooks.append(m.register_forward_hook(lambda mod, i, o, nama=nama: log.append((nama, type(mod).__name__, tuple(o.shape[1:])))))
    model.eval()
    model(torch.randn(*input_size, device=next(model.parameters()).device))
    for h in hooks:
        h.remove()
    return log

## 4. Perancangan dan Implementasi Arsitektur
Semua arsitektur ditulis sendiri dengan `nn.Module`. Ringkasan adaptasi untuk citra 32×32:

| Model | Tahun | Ide kunci (slide) | Yang dipertahankan | Yang diadaptasi untuk 32×32 |
|---|---|---|---|---|
| LeNet-5 | 1998 | conv → pool → FC (P4) | Seluruh arsitektur: Tanh + AvgPool, 6/16 filter 5×5, FC 120-84 | C1 menerima 3 kanal (RGB) |
| AlexNet | 2012 | ReLU, Dropout, max-pool overlap (sl. 7–11) | 5 conv + 3 FC, kanal 64-192-384-256-256, FC 4096-4096, Dropout 0,5 | conv1 11×11 stride 4 → 5×5 stride 1 |
| VGG-16 | 2014 | Tumpukan conv 3×3 seragam (sl. 12–15) | 13 conv 3×3 dalam 5 blok, FC 4096-4096 | FC6 menerima 512 (bukan 7×7×512); **BatchNorm** ditambahkan (varian `vgg16_bn`, sl. 15) |
| GoogLeNet | 2014 | Inception, bottleneck 1×1, GAP (sl. 16–17) | 9 modul Inception dengan kanal paper (termasuk cabang 5×5 asli), GAP | Stem 7×7/s2 + 2 max-pool → 1 conv 3×3; BatchNorm; tanpa auxiliary classifier |
| ResNet-18 | 2015 | Residual connection (sl. 20–25) | 4 stage × 2 basic block, kanal 64-512, projection shortcut, GAP | Stem 7×7/s2 + max-pool → conv 3×3/s1 |
| MobileNetV2 | 2018 | Depthwise separable, inverted residual (sl. 28–31) | Konfigurasi blok (t, c, n, s) paper, ReLU6, linear bottleneck | Stride stem dan stage kanal-24: 2 → 1 |

**Mengapa VGG-16 memakai BatchNorm?** Slide 19 menyebut VGG tanpa BN harus dilatih bertahap (11 → 16 layer). Dilatih dari nol sekaligus dengan Adam lr 0,001, VGG-16 polos berisiko macet di akurasi ±10%. Varian dengan BN (`vgg16_bn`) tetap memakai tumpukan 3×3 yang sama, jadi ciri khas VGG tidak hilang.

### 4.1 LeNet-5 (1998)

In [ ]:
class LeNet5(nn.Module):
    """LeNet-5 (LeCun, 1998) — versi klasik notebook 02: Tanh + AvgPool.
    Satu-satunya perubahan: C1 menerima 3 kanal (RGB), bukan 1 kanal."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5), nn.Tanh(),       # C1: 32 -> 28
            nn.AvgPool2d(2, 2),                               # S2: 28 -> 14
            nn.Conv2d(6, 16, kernel_size=5), nn.Tanh(),      # C3: 14 -> 10
            nn.AvgPool2d(2, 2),                               # S4: 10 -> 5
        )
        self.classifier = nn.Sequential(
            nn.Linear(16 * 5 * 5, 120), nn.Tanh(),            # C5
            nn.Linear(120, 84), nn.Tanh(),                    # F6
            nn.Linear(84, num_classes),                       # output (logits)
        )

    def forward(self, x):
        return self.classifier(torch.flatten(self.feature_extractor(x), 1))

### 4.2 AlexNet (2012)

In [ ]:
class AlexNet(nn.Module):
    """AlexNet (2012) versi CIFAR. Dipertahankan: 5 conv + 3 FC, ReLU, max-pool 3x3 stride 2
    (overlapping), Dropout 0.5, FC 4096-4096, kanal 64-192-384-256-256 (versi torchvision).
    Diadaptasi: conv1 11x11/s4 -> 5x5/s1 dan conv2 tetap 5x5, karena input hanya 32x32."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2), nn.ReLU(inplace=True),   # 32
            nn.MaxPool2d(3, 2),                                                            # 32 -> 15
            nn.Conv2d(64, 192, kernel_size=5, padding=2), nn.ReLU(inplace=True),          # 15
            nn.MaxPool2d(3, 2),                                                            # 15 -> 7
            nn.Conv2d(192, 384, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),         # 7
            nn.MaxPool2d(3, 2),                                                            # 7 -> 3
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256 * 3 * 3, 4096), nn.ReLU(inplace=True),      # FC6
            nn.Dropout(0.5), nn.Linear(4096, 4096), nn.ReLU(inplace=True),             # FC7
            nn.Linear(4096, num_classes),                                              # FC8
        )

    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

### 4.3 VGG-16 (2014)

In [ ]:
VGG16_CFG = [(2, 64), (2, 128), (3, 256), (3, 512), (3, 512)]   # (jumlah conv, kanal) per blok

def vgg_block(c_in, c_out, n_conv):
    """Blok VGG (slide 15) + BatchNorm setelah tiap conv (varian vgg16_bn)."""
    layers = []
    for i in range(n_conv):
        c = c_in if i == 0 else c_out
        layers += [nn.Conv2d(c, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]
    layers.append(nn.MaxPool2d(2, 2))                     # resolusi / 2
    return nn.Sequential(*layers)

class VGG16(nn.Module):
    """VGG-16-BN (2014) versi CIFAR. 13 conv 3x3 identik dengan paper; setelah 5 blok,
    32x32 menjadi 1x1x512 sehingga FC6 menerima 512 (bukan 7x7x512)."""
    def __init__(self, num_classes=10):
        super().__init__()
        blok, c_in = [], 3
        for n_conv, c_out in VGG16_CFG:
            blok.append(vgg_block(c_in, c_out, n_conv))
            c_in = c_out
        self.features = nn.Sequential(*blok)
        self.classifier = nn.Sequential(
            nn.Linear(512, 4096), nn.ReLU(inplace=True), nn.Dropout(0.5),    # FC6
            nn.Linear(4096, 4096), nn.ReLU(inplace=True), nn.Dropout(0.5),   # FC7
            nn.Linear(4096, num_classes),                                    # FC8
        )

    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))

### 4.4 GoogLeNet (2014)

In [ ]:
class BasicConv2d(nn.Module):
    """Conv (tanpa bias) -> BatchNorm -> ReLU (pola baku slide 19)."""
    def __init__(self, c_in, c_out, **kwargs):
        super().__init__()
        self.conv = nn.Conv2d(c_in, c_out, bias=False, **kwargs)
        self.bn = nn.BatchNorm2d(c_out, eps=0.001)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)), inplace=True)

class Inception(nn.Module):
    """Modul Inception (slide 17): 4 cabang paralel, digabung di dimensi kanal."""
    def __init__(self, c_in, c1, c3_red, c3, c5_red, c5, c_pool):
        super().__init__()
        self.branch1 = BasicConv2d(c_in, c1, kernel_size=1)
        self.branch2 = nn.Sequential(BasicConv2d(c_in, c3_red, kernel_size=1),          # 1x1 bottleneck
                                     BasicConv2d(c3_red, c3, kernel_size=3, padding=1))
        self.branch3 = nn.Sequential(BasicConv2d(c_in, c5_red, kernel_size=1),          # 1x1 bottleneck
                                     BasicConv2d(c5_red, c5, kernel_size=5, padding=2))  # 5x5 sesuai paper
        self.branch4 = nn.Sequential(nn.MaxPool2d(3, stride=1, padding=1),
                                     BasicConv2d(c_in, c_pool, kernel_size=1))

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)

class GoogLeNet(nn.Module):
    """GoogLeNet (2014) versi CIFAR. 9 modul Inception dengan konfigurasi kanal paper + GAP.
    Diadaptasi: stem 7x7/s2 + 2 max-pool diganti satu conv 3x3 (resolusi tetap 32x32),
    BatchNorm ditambahkan, auxiliary classifier tidak dipakai."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = BasicConv2d(3, 192, kernel_size=3, padding=1)                 # 32x32x192
        self.inception3a = Inception(192, 64, 96, 128, 16, 32, 32)               # -> 256
        self.inception3b = Inception(256, 128, 128, 192, 32, 96, 64)             # -> 480
        self.maxpool3 = nn.MaxPool2d(3, stride=2, padding=1)                     # 32 -> 16
        self.inception4a = Inception(480, 192, 96, 208, 16, 48, 64)              # -> 512
        self.inception4b = Inception(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = Inception(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = Inception(512, 112, 144, 288, 32, 64, 64)             # -> 528
        self.inception4e = Inception(528, 256, 160, 320, 32, 128, 128)           # -> 832
        self.maxpool4 = nn.MaxPool2d(3, stride=2, padding=1)                     # 16 -> 8
        self.inception5a = Inception(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = Inception(832, 384, 192, 384, 48, 128, 128)           # -> 1024
        self.avgpool = nn.AdaptiveAvgPool2d(1)                                   # Global Average Pooling
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.maxpool3(self.inception3b(self.inception3a(x)))
        x = self.inception4c(self.inception4b(self.inception4a(x)))
        x = self.maxpool4(self.inception4e(self.inception4d(x)))
        x = self.inception5b(self.inception5a(x))
        return self.fc(self.dropout(torch.flatten(self.avgpool(x), 1)))

### 4.5 ResNet-18 (2015) — model pilihan 1

In [ ]:
class BasicBlock(nn.Module):
    """Basic block ResNet (slide 23): y = ReLU(F(x) + shortcut(x))."""
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:                           # projection shortcut
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False),
                                          nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))                      # <-- skip connection

class ResNet18(nn.Module):
    """ResNet-18 (2015) versi CIFAR. 4 stage x 2 basic block, kanal 64-128-256-512, GAP + FC.
    Diadaptasi: stem 7x7/s2 + max-pool diganti conv 3x3/s1 (seperti eksperimen CIFAR di paper ResNet)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.layer1 = nn.Sequential(BasicBlock(64, 64), BasicBlock(64, 64))            # 32x32
        self.layer2 = nn.Sequential(BasicBlock(64, 128, 2), BasicBlock(128, 128))      # 16x16
        self.layer3 = nn.Sequential(BasicBlock(128, 256, 2), BasicBlock(256, 256))     # 8x8
        self.layer4 = nn.Sequential(BasicBlock(256, 512, 2), BasicBlock(512, 512))     # 4x4
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        return self.fc(torch.flatten(self.avgpool(x), 1))

### 4.6 MobileNetV2 (2018) — model pilihan 2

In [ ]:
class InvertedResidual(nn.Module):
    """Blok MobileNetV2 (slide 31): 1x1 ekspansi -> 3x3 depthwise -> 1x1 proyeksi LINEAR.
    Skip connection hanya bila stride 1 dan kanal masuk = kanal keluar."""
    def __init__(self, c_in, c_out, stride, t):
        super().__init__()
        hidden = c_in * t
        self.use_skip = stride == 1 and c_in == c_out
        layers = []
        if t != 1:
            layers += [nn.Conv2d(c_in, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.ReLU6(inplace=True)]
        layers += [
            nn.Conv2d(hidden, hidden, 3, stride, 1, groups=hidden, bias=False),   # depthwise: groups = kanal
            nn.BatchNorm2d(hidden), nn.ReLU6(inplace=True),
            nn.Conv2d(hidden, c_out, 1, bias=False), nn.BatchNorm2d(c_out),        # pointwise, tanpa aktivasi
        ]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.block(x) if self.use_skip else self.block(x)

class MobileNetV2(nn.Module):
    """MobileNetV2 (2018) versi CIFAR. Konfigurasi blok (t, c, n, s) identik dengan paper.
    Diadaptasi: stride stem 2 -> 1 dan stride stage kanal-24 2 -> 1, sehingga peta fitur akhir 4x4."""
    CFG = [(1, 16, 1, 1), (6, 24, 2, 1), (6, 32, 3, 2), (6, 64, 4, 2),
           (6, 96, 3, 1), (6, 160, 3, 2), (6, 320, 1, 1)]

    def __init__(self, num_classes=10):
        super().__init__()
        layers = [nn.Sequential(nn.Conv2d(3, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU6(inplace=True))]
        c_in = 32
        for t, c, n, s in self.CFG:
            for i in range(n):
                layers.append(InvertedResidual(c_in, c, s if i == 0 else 1, t))
                c_in = c
        layers.append(nn.Sequential(nn.Conv2d(c_in, 1280, 1, bias=False), nn.BatchNorm2d(1280), nn.ReLU6(inplace=True)))
        self.features = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, num_classes))

    def forward(self, x):
        return self.classifier(torch.flatten(self.avgpool(self.features(x)), 1))

In [ ]:
PEMBUAT_MODEL = {
    "LeNet-5": LeNet5, "AlexNet": AlexNet, "VGG-16": VGG16,
    "GoogLeNet": GoogLeNet, "ResNet-18": ResNet18, "MobileNetV2": MobileNetV2,
}
INFO_MODEL = {
    "LeNet-5":     (1998, "conv-pool-FC klasik, Tanh"),
    "AlexNet":     (2012, "ReLU, Dropout, model besar"),
    "VGG-16":      (2014, "tumpukan conv 3x3 seragam"),
    "GoogLeNet":   (2014, "Inception, 1x1, GAP"),
    "ResNet-18":   (2015, "residual connection"),
    "MobileNetV2": (2018, "depthwise separable, inverted residual"),
}

### 4.7 Verifikasi arsitektur: shape, parameter, dan komputasi
Sebelum training, setiap model diuji dengan input dummy 32×32×3. Output harus `(1, 10)`. LeNet-5 harus tepat **62.006** parameter (61.706 di slide + 300, karena C1 menerima 3 kanal).

In [ ]:
assert hitung_param(LeNet5()) == 62_006, "LeNet-5 tidak sesuai hitungan analitis"

ringkasan_arsitektur = {}
for nama in MODEL_DIPAKAI:
    m = PEMBUAT_MODEL[nama]()
    p, pfc = hitung_param(m), hitung_param_fc(m)
    mac = hitung_mac(m)
    out = m(torch.randn(1, 3, 32, 32)).shape
    assert tuple(out) == (1, 10)
    ringkasan_arsitektur[nama] = dict(param=p, param_fc=pfc, mac=mac)
    jejak = jejak_shape(m)
    print(f"\n=== {nama} | {p:,} parameter | {mac/1e6:,.1f} juta MAC | {pfc/p:.1%} parameter di FC ===")
    for i in range(0, len(jejak), 3):
        print("   " + " | ".join(f"{n} -> {'x'.join(map(str, s))}" for n, _, s in jejak[i:i + 3]))
    del m

## 5. Pelatihan
Fungsi pelatihan memakai **5 langkah baku** (slide P4 / notebook 02): `zero_grad` → forward → loss → `backward` → `step`.
Setiap model dimulai dari kondisi yang sama: `set_seed(42)` sebelum model dibuat, generator shuffle dengan seed 42, optimizer `Adam(lr=0.001)`, `CrossEntropyLoss`, batch 64, 10 epoch.

Karena pembagian default torchvision hanya terdiri dari *train* dan *test*, akurasi uji dicatat di setiap epoch untuk **memantau kurva belajar**. Metrik resmi perbandingan adalah **akurasi uji pada epoch ke-10**, bukan akurasi tertinggi. Memilih epoch terbaik berdasarkan data uji sama saja dengan "mengintip" data uji.

In [ ]:
def autocast():
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)

@torch.no_grad()
def evaluasi(model, loader, criterion):
    model.eval()
    total_loss, benar, total, semua_pred = 0.0, 0, 0, []
    for x, y in loader:
        with autocast():
            out = model(x)
        total_loss += criterion(out.float(), y).item() * y.size(0)
        pred = out.argmax(1)
        benar += (pred == y).sum().item()
        total += y.size(0)
        semua_pred.append(pred.cpu())
    return total_loss / total, benar / total, torch.cat(semua_pred)

def latih_model(nama):
    set_seed(CONFIG["SEED"])
    model = PEMBUAT_MODEL[nama]().to(device)
    train_loader, test_loader = buat_loader(CONFIG["SEED"])
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

    riwayat = []
    print(f"\n{'=' * 78}\n{nama}: {hitung_param(model):,} parameter\n{'=' * 78}")
    for epoch in range(1, CONFIG["EPOCHS"] + 1):
        model.train()
        t0 = time.time()
        total_loss, benar, total = 0.0, 0, 0
        for x, y in train_loader:
            optimizer.zero_grad(set_to_none=True)      # 1. reset gradien
            with autocast():
                out = model(x)                         # 2. forward
                loss = criterion(out, y)               # 3. hitung loss
            scaler.scale(loss).backward()              # 4. backward
            scaler.step(optimizer)                     # 5. update bobot
            scaler.update()
            total_loss += loss.item() * y.size(0)
            benar += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        if device.type == "cuda":
            torch.cuda.synchronize()
        detik = time.time() - t0
        test_loss, test_acc, pred = evaluasi(model, test_loader, criterion)
        riwayat.append(dict(epoch=epoch, train_loss=total_loss / total, train_acc=benar / total,
                            test_loss=test_loss, test_acc=test_acc, detik=detik))
        print(f"Epoch {epoch:2d}/{CONFIG['EPOCHS']} | train loss {total_loss/total:.4f} acc {benar/total:6.2%} "
              f"| test loss {test_loss:.4f} acc {test_acc:6.2%} | {detik:5.1f}s")

    hasil = dict(nama=nama, riwayat=riwayat, pred_test=pred.tolist(), **ringkasan_arsitektur[nama])
    with open(os.path.join(CONFIG["DIR_HASIL"], f"{nama}.json"), "w") as f:
        json.dump(hasil, f)
    del model, optimizer
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return hasil

Jalankan pelatihan keenam model. Bila runtime terputus di tengah jalan, jalankan ulang sel ini. Model yang hasilnya sudah tersimpan di folder `hasil/` akan dimuat, tidak dilatih ulang.

In [ ]:
HASIL = {}
t_mulai = time.time()
for nama in MODEL_DIPAKAI:
    path = os.path.join(CONFIG["DIR_HASIL"], f"{nama}.json")
    if os.path.exists(path):
        with open(path) as f:
            HASIL[nama] = json.load(f)
        print(f"{nama}: hasil tersimpan dimuat dari {path}")
    else:
        HASIL[nama] = latih_model(nama)
print(f"\nTotal waktu: {(time.time() - t_mulai) / 60:.1f} menit")

## 6. Tabel Perbandingan

In [ ]:
baris = []
for nama in MODEL_DIPAKAI:
    h = HASIL[nama]
    r = h["riwayat"]
    akhir = r[-1]
    terbaik = max(r, key=lambda e: e["test_acc"])
    baris.append({
        "Model": nama,
        "Tahun": INFO_MODEL[nama][0],
        "Ide kunci": INFO_MODEL[nama][1],
        "Parameter (juta)": h["param"] / 1e6,
        "% param di FC": 100 * h["param_fc"] / h["param"],
        "MAC (juta)": h["mac"] / 1e6,
        "Ukuran (MB)": h["param"] * 4 / 1e6,
        "Waktu/epoch (s)": float(np.mean([e["detik"] for e in r])),
        "Train acc (%)": 100 * akhir["train_acc"],
        "Test acc (%)": 100 * akhir["test_acc"],
        "Gap train-test": 100 * (akhir["train_acc"] - akhir["test_acc"]),
        "Test acc maks (%)": 100 * terbaik["test_acc"],
        "Epoch maks": terbaik["epoch"],
    })
tabel = pd.DataFrame(baris)
tabel.to_csv(os.path.join(CONFIG["DIR_HASIL"], "tabel_perbandingan.csv"), index=False)

pd.set_option("display.width", 200)
tampil = tabel.drop(columns=["Ide kunci"]).sort_values("Test acc (%)", ascending=False)
print(f"Diurutkan berdasarkan akurasi uji epoch ke-{CONFIG['EPOCHS']} (metrik resmi):\n")
print(tampil.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

## 7. Visualisasi Hasil
### 7.1 Kurva belajar

In [ ]:
ep = range(1, CONFIG["EPOCHS"] + 1)
fig, axs = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
panel = [("train_loss", "Train loss", axs[0, 0]), ("test_loss", "Test loss", axs[0, 1]),
         ("train_acc", "Train accuracy", axs[1, 0]), ("test_acc", "Test accuracy", axs[1, 1])]
for kunci, judul, ax in panel:
    for nama in MODEL_DIPAKAI:
        nilai = [e[kunci] for e in HASIL[nama]["riwayat"]]
        if "acc" in kunci:
            nilai = [100 * v for v in nilai]
        ax.plot(ep, nilai, marker=PENANDA[nama], ms=5, lw=2, color=WARNA[nama], label=nama)
    ax.set_title(judul)
    ax.set_ylabel("%" if "acc" in kunci else "cross-entropy")
for ax in axs[1]:
    ax.set_xlabel("Epoch")
    ax.set_xticks(list(ep))
h, l = axs[0, 0].get_legend_handles_labels()
plt.suptitle("Kurva belajar 6 arsitektur CNN (Adam lr 0,001, batch 64, seed 42)", y=1.0)
plt.tight_layout()
fig.legend(h, l, loc="lower center", ncol=6, frameon=False, bbox_to_anchor=(0.5, -0.04))
plt.savefig(os.path.join(CONFIG["DIR_HASIL"], "kurva_belajar.png"), bbox_inches="tight")
plt.show()

### 7.2 Akurasi uji dan efisiensi

In [ ]:
urut = tabel.sort_values("Test acc (%)")
# posisi label bergantian agar AlexNet & VGG-16 (parameter berdekatan) tidak bertumpuk
OFFSET_LABEL = {"LeNet-5": (7, 4, "left"), "AlexNet": (-7, -13, "right"), "VGG-16": (7, 6, "left"),
                "GoogLeNet": (7, 4, "left"), "ResNet-18": (7, -12, "left"), "MobileNetV2": (7, 4, "left")}
fig, axs = plt.subplots(1, 3, figsize=(16, 4.6))

ax = axs[0]
bar = ax.barh(urut["Model"], urut["Test acc (%)"], color="#2a78d6", height=0.6)
for b, v in zip(bar, urut["Test acc (%)"]):
    ax.text(v + 0.5, b.get_y() + b.get_height() / 2, f"{v:.2f}%", va="center", fontsize=9)
ax.set_xlim(0, 100)
ax.set_title(f"Akurasi uji (epoch {CONFIG['EPOCHS']})")
ax.set_xlabel("%")
ax.grid(axis="y", visible=False)

for ax, kolom, judul in [(axs[1], "Parameter (juta)", "Akurasi vs jumlah parameter"),
                         (axs[2], "MAC (juta)", "Akurasi vs komputasi per citra")]:
    for _, r in tabel.iterrows():
        ax.scatter(r[kolom], r["Test acc (%)"], s=70, color="#2a78d6", marker=PENANDA[r["Model"]], zorder=3)
        dx, dy, ha = OFFSET_LABEL[r["Model"]]
        ax.annotate(r["Model"], (r[kolom], r["Test acc (%)"]), xytext=(dx, dy), textcoords="offset points",
                    fontsize=9, ha=ha)
    ax.set_xscale("log")
    ax.set_title(judul)
    ax.set_xlabel(kolom + " — skala log")
    ax.set_ylabel("Akurasi uji (%)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["DIR_HASIL"], "akurasi_efisiensi.png"), bbox_inches="tight")
plt.show()

### 7.3 Distribusi parameter: konvolusi vs fully connected

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
conv_pct = 100 - tabel["% param di FC"]
ax.barh(tabel["Model"], conv_pct, color="#2a78d6", height=0.6, label="Konvolusi (+ BN)")
ax.barh(tabel["Model"], tabel["% param di FC"], left=conv_pct, color="#eb6834", height=0.6, label="Fully connected")
for i, (c, p) in enumerate(zip(conv_pct, tabel["Parameter (juta)"])):
    ax.text(101, i, f"{p:.2f} jt", va="center", fontsize=9)
ax.set_xlim(0, 112)
ax.invert_yaxis()
ax.set_xlabel("% dari total parameter")
ax.set_title("Di mana parameter tiap model berada?")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9, frameon=False)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["DIR_HASIL"], "distribusi_parameter.png"), bbox_inches="tight")
plt.show()

## 8. Analisis per Kelas
### 8.1 Akurasi per kelas untuk setiap model

In [ ]:
y_np = y_test.numpy()
akurasi_kelas = np.array([[(np.array(HASIL[n]["pred_test"])[y_np == c] == c).mean() * 100 for c in range(10)]
                          for n in MODEL_DIPAKAI])
fig, ax = plt.subplots(figsize=(13, 4.2))
im = ax.imshow(akurasi_kelas, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(10), KELAS, rotation=30)
ax.set_yticks(range(len(MODEL_DIPAKAI)), MODEL_DIPAKAI)
ax.grid(False)
for i in range(len(MODEL_DIPAKAI)):
    for j in range(10):
        v = akurasi_kelas[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9, color="white" if v > 60 else "#0b0b0b")
fig.colorbar(im, ax=ax, label="akurasi (%)")
ax.set_title("Akurasi uji per kelas (%)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["DIR_HASIL"], "akurasi_per_kelas.png"), bbox_inches="tight")
plt.show()

rata_kelas = akurasi_kelas.mean(axis=0)
print("Kelas tersulit (rata-rata 6 model):", ", ".join(f"{KELAS[i]} ({rata_kelas[i]:.1f}%)" for i in rata_kelas.argsort()[:3]))
print("Kelas termudah (rata-rata 6 model):", ", ".join(f"{KELAS[i]} ({rata_kelas[i]:.1f}%)" for i in rata_kelas.argsort()[::-1][:3]))

### 8.2 Confusion matrix: model terbaik vs model terlemah

In [ ]:
def confusion(pred):
    cm = np.zeros((10, 10), dtype=int)
    for t, p in zip(y_np, pred):
        cm[t, p] += 1
    return cm

terbaik = tabel.loc[tabel["Test acc (%)"].idxmax(), "Model"]
terlemah = tabel.loc[tabel["Test acc (%)"].idxmin(), "Model"]
fig, axs = plt.subplots(1, 2, figsize=(15, 6.2))
for ax, nama in zip(axs, [terbaik, terlemah]):
    cm = confusion(HASIL[nama]["pred_test"])
    ax.imshow(cm, cmap="Blues")
    ax.grid(False)
    ax.set_xticks(range(10), KELAS, rotation=45, ha="right")
    ax.set_yticks(range(10), KELAS)
    for i in range(10):
        for j in range(10):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "#0b0b0b")
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Label sebenarnya")
    ax.set_title(f"{nama} — akurasi {np.trace(cm) / cm.sum():.2%}")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["DIR_HASIL"], "confusion_matrix.png"), bbox_inches="tight")
plt.show()

for nama in [terbaik, terlemah]:
    cm = confusion(HASIL[nama]["pred_test"])
    np.fill_diagonal(cm, 0)
    i, j = np.unravel_index(cm.argmax(), cm.shape)
    print(f"{nama}: kesalahan terbanyak = '{KELAS[i]}' diprediksi sebagai '{KELAS[j]}' ({cm[i, j]} citra)")

## 9. Ringkasan Temuan (dihitung otomatis dari hasil)
Bagian ini hanya menyajikan **fakta numerik**. Interpretasinya ada di Bagian 10.

In [ ]:
t = tabel.set_index("Model")
acc = t["Test acc (%)"]
E = CONFIG["EPOCHS"]
print(f"1. Peringkat akurasi uji (epoch {E}):")
for i, (n, v) in enumerate(acc.sort_values(ascending=False).items(), 1):
    print(f"   {i}. {n:<12} {v:6.2f}%  ({t.loc[n, 'Parameter (juta)']:.2f} jt parameter, {t.loc[n, 'MAC (juta)']:.0f} jt MAC)")

base = acc["LeNet-5"]
print(f"\n2. Dibanding baseline LeNet-5 ({base:.2f}%):")
for n in MODEL_DIPAKAI[1:]:
    print(f"   {n:<12} {acc[n] - base:+6.2f} poin")

eff = acc / t["Parameter (juta)"]
best_eff = eff.drop("LeNet-5").idxmax()
print(f"\n3. Efisiensi parameter (di luar LeNet-5): {best_eff} memberi akurasi terbanyak per juta parameter "
      f"({acc[best_eff]:.2f}% dengan {t.loc[best_eff, 'Parameter (juta)']:.2f} jt).")
n_max, n_min = t["Parameter (juta)"].idxmax(), t.drop("LeNet-5")["Parameter (juta)"].idxmin()
print(f"   {n_max} punya {t.loc[n_max, 'Parameter (juta)'] / t.loc[n_min, 'Parameter (juta)']:.0f}x parameter {n_min}; "
      f"selisih akurasinya {acc[n_max] - acc[n_min]:+.2f} poin.")

print(f"\n4. Overfitting (gap train - test pada epoch {E}):")
for n, g in t["Gap train-test"].sort_values(ascending=False).items():
    label = "overfitting jelas" if g > 10 else ("overfitting ringan" if g > 3 else "belum overfitting")
    print(f"   {n:<12} {g:+6.2f} poin -> {label}")

print(f"\n5. Apakah model masih belajar di akhir? (perubahan akurasi uji epoch {max(1, E-2)} -> {E})")
for n in MODEL_DIPAKAI:
    r = HASIL[n]["riwayat"]
    d = 100 * (r[-1]["test_acc"] - r[max(0, len(r) - 3)]["test_acc"])
    print(f"   {n:<12} {d:+6.2f} poin -> " + (f"masih naik: {E} epoch belum cukup" if d > 1 else "sudah mendatar / berfluktuasi"))

gagal = [n for n in MODEL_DIPAKAI if acc[n] < 15]
print("\n6. Model yang gagal belajar (akurasi uji < 15%, setara tebakan acak 10%):")
print("   " + (", ".join(gagal) + " -> bahas penyebabnya di Bagian 10 (mis. tanpa BatchNorm + Adam lr 0,001)" if gagal else "tidak ada"))

print("\n7. Biaya waktu training (rata-rata per epoch):")
for n, s in t["Waktu/epoch (s)"].sort_values().items():
    print(f"   {n:<12} {s:6.1f} s")

## 9b. Analisis 4 Model Utama: LeNet-5, AlexNet, VGG-16, GoogLeNet
Bagian ini menjawab empat pertanyaan: **(1)** model mana yang paling akurat, **(2)** mana yang paling efisien, **(3)** apa trade-off-nya, dan **(4)** model mana yang direkomendasikan untuk skenario apa.

**Apa arti "efisien"?** Biaya sebuah model punya beberapa dimensi yang berbeda, dan satu model bisa murah di satu dimensi tetapi mahal di dimensi lain:

| Dimensi biaya | Diukur dengan | Penting untuk |
|---|---|---|
| Memori / penyimpanan | Parameter, ukuran file (MB) | Ponsel, IoT, unduhan aplikasi |
| Komputasi | MAC per citra | Baterai, panas perangkat |
| Kecepatan inferensi | Latensi CPU (1 citra) | Aplikasi real-time |
| Biaya training | Waktu per epoch di GPU | Eksperimen, retraining |

Rasio sederhana seperti "akurasi ÷ parameter" **menyesatkan**: LeNet-5 akan selalu menang karena parameternya sangat sedikit, walaupun akurasinya rendah. Karena itu dipakai dua cara yang lebih jujur:
1. **Pareto-optimal:** sebuah model efisien jika **tidak ada model lain yang lebih akurat sekaligus lebih murah**.
2. **Biaya per poin akurasi di atas baseline LeNet-5:** berapa juta parameter (atau MAC) yang "dibayar" untuk setiap poin akurasi tambahan.

In [ ]:
EMPAT = ["LeNet-5", "AlexNet", "VGG-16", "GoogLeNet"]

@torch.no_grad()
def latensi_cpu_ms(model, ulang=30):
    """Median latensi inferensi 1 citra di 1 thread CPU (proksi perangkat edge). Bobot tidak berpengaruh."""
    n_thread = torch.get_num_threads()
    torch.set_num_threads(1)
    model = model.cpu().eval()
    x = torch.randn(1, 3, 32, 32)
    for _ in range(5):
        model(x)
    waktu = []
    for _ in range(ulang):
        t0 = time.perf_counter()
        model(x)
        waktu.append((time.perf_counter() - t0) * 1000)
    torch.set_num_threads(n_thread)
    return float(np.median(waktu))

a4 = tabel.set_index("Model").loc[EMPAT].copy()
a4["Latensi CPU (ms)"] = [latensi_cpu_ms(PEMBUAT_MODEL[n]()) for n in EMPAT]
kolom = ["Test acc (%)", "Gap train-test", "Parameter (juta)", "Ukuran (MB)", "MAC (juta)",
         "Latensi CPU (ms)", "Waktu/epoch (s)"]
print("Perbandingan 4 model utama (diurutkan berdasarkan akurasi uji):\n")
print(a4[kolom].sort_values("Test acc (%)", ascending=False).to_string(float_format=lambda v: f"{v:,.2f}"))

In [ ]:
acc4 = a4["Test acc (%)"]
BIAYA = {"Ukuran (MB)": "memori", "MAC (juta)": "komputasi", "Latensi CPU (ms)": "latensi CPU", "Waktu/epoch (s)": "waktu training"}

print("1) AKURASI TESTING")
for i, (n, v) in enumerate(acc4.sort_values(ascending=False).items(), 1):
    print(f"   {i}. {n:<10} {v:6.2f}%  (gap train-test {a4.loc[n, 'Gap train-test']:+.2f} poin)")
selisih = acc4.max() - acc4.drop(acc4.idxmax()).max()
print(f"   Selisih juara 1 dan 2: {selisih:.2f} poin" + ("  -> < 1 poin, belum tentu berbeda nyata (1 seed)" if selisih < 1 else ""))

print("\n2) EFISIENSI: model Pareto-optimal per dimensi biaya")
for k, label in BIAYA.items():
    pareto = [n for n in EMPAT if not any(acc4[m] >= acc4[n] and a4.loc[m, k] <= a4.loc[n, k]
                                         and (acc4[m] > acc4[n] or a4.loc[m, k] < a4.loc[n, k]) for m in EMPAT)]
    print(f"   {label:<15}: {', '.join(pareto)}")

print("\n   Biaya per 1 poin akurasi di atas LeNet-5:")
base = acc4["LeNet-5"]
for n in EMPAT[1:]:
    naik = acc4[n] - base
    if naik <= 0:
        print(f"   {n:<10} tidak lebih akurat dari LeNet-5")
        continue
    print(f"   {n:<10} +{naik:5.2f} poin | {a4.loc[n, 'Ukuran (MB)'] / naik:6.2f} MB/poin | "
          f"{a4.loc[n, 'MAC (juta)'] / naik:6.1f} jt MAC/poin | {a4.loc[n, 'Latensi CPU (ms)'] / naik:5.2f} ms/poin")

print("\n3) TRADE-OFF")
g, al, v = "GoogLeNet", "AlexNet", "VGG-16"
print(f"   GoogLeNet vs VGG-16 : {a4.loc[v, 'Parameter (juta)'] / a4.loc[g, 'Parameter (juta)']:.1f}x lebih sedikit parameter, "
      f"tetapi {a4.loc[g, 'MAC (juta)'] / a4.loc[v, 'MAC (juta)']:.1f}x MAC dan {a4.loc[g, 'Latensi CPU (ms)'] / a4.loc[v, 'Latensi CPU (ms)']:.1f}x latensi CPU; "
      f"selisih akurasi {acc4[g] - acc4[v]:+.2f} poin")
print(f"   VGG-16 vs AlexNet   : {a4.loc[v, 'MAC (juta)'] / a4.loc[al, 'MAC (juta)']:.1f}x MAC, "
      f"{a4.loc[v, 'Parameter (juta)'] / a4.loc[al, 'Parameter (juta)']:.2f}x parameter; selisih akurasi {acc4[v] - acc4[al]:+.2f} poin")
print(f"   AlexNet vs LeNet-5  : {a4.loc[al, 'Parameter (juta)'] / a4.loc['LeNet-5', 'Parameter (juta)']:.0f}x parameter; "
      f"selisih akurasi {acc4[al] - acc4['LeNet-5']:+.2f} poin")

print("\n4) REKOMENDASI (kandidat layak = akurasi maksimal 5 poin di bawah model terbaik)")
layak = [n for n in EMPAT if acc4[n] >= acc4.max() - 5]
pilih = lambda k: min(layak, key=lambda n: a4.loc[n, k])
print(f"   Kandidat layak               : {', '.join(layak)}")
print(f"   Server / akurasi maksimum    : {acc4.idxmax()} ({acc4.max():.2f}%)")
print(f"   Ponsel, memori terbatas      : {pilih('Ukuran (MB)')} ({a4.loc[pilih('Ukuran (MB)'), 'Ukuran (MB)']:.1f} MB)")
print(f"   Real-time di CPU             : {pilih('Latensi CPU (ms)')} ({a4.loc[pilih('Latensi CPU (ms)'), 'Latensi CPU (ms)']:.1f} ms/citra)")
print(f"   Eksperimen / training cepat  : {pilih('Waktu/epoch (s)')} ({a4.loc[pilih('Waktu/epoch (s)'), 'Waktu/epoch (s)']:.1f} s/epoch)")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, k in zip(axs, ["Ukuran (MB)", "MAC (juta)", "Latensi CPU (ms)"]):
    for n in EMPAT:
        ax.scatter(a4.loc[n, k], acc4[n], s=80, color=WARNA[n], marker=PENANDA[n], zorder=3)
        ax.annotate(n, (a4.loc[n, k], acc4[n]), xytext=(7, 4), textcoords="offset points", fontsize=9)
    ax.set_xscale("log")
    ax.set_xlabel(k + " — skala log (kiri = lebih murah)")
    ax.set_ylabel("Akurasi uji (%)")
    ax.set_title(f"Akurasi vs {BIAYA[k]}")
plt.suptitle("Trade-off 4 model utama: model ideal berada di kiri atas", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["DIR_HASIL"], "tradeoff_4_model.png"), bbox_inches="tight")
plt.show()

### Pembahasan 4 model utama
*(Isi titik-titik dengan angka dari dua sel di atas.)*

**Akurasi testing.** Model paling akurat adalah ...... (......%), disusul ...... (......%). LeNet-5 berada di posisi terakhir (......%) karena kapasitasnya terlalu kecil untuk CIFAR-10.

**Efisiensi.** Dilihat dari memori, ...... paling efisien: ...... MB, ±5× lebih kecil dari AlexNet dan VGG-16, berkat bottleneck 1×1 dan Global Average Pooling. Dilihat dari komputasi dan latensi, hasilnya **terbalik**: GoogLeNet butuh ±1.447 juta MAC (±7× AlexNet) dan latensi CPU ±...... ms. Penyebabnya, setiap bobot konvolusi dipakai ulang di setiap posisi piksel (±246 MAC per parameter), sedangkan bobot FC di AlexNet/VGG hanya dipakai sekali per citra (±7–10 MAC per parameter).

**Trade-off.** Tidak ada satu pun dari empat model klasik ini yang murah di **semua** dimensi:
- AlexNet dan VGG-16 cepat dihitung tetapi **boros memori** (115–135 MB, sebagian besar di layer FC).
- GoogLeNet hemat memori tetapi **berat dihitung**.
- LeNet-5 murah di semua dimensi tetapi **tidak cukup akurat**.

Keterbatasan inilah yang mendorong lahirnya arsitektur seperti MobileNet, yang dirancang untuk hemat memori **dan** komputasi sekaligus.

**Rekomendasi.**
- Server / akurasi maksimum: ...... karena ......
- Ponsel dengan memori terbatas: ...... karena ......
- Aplikasi real-time di CPU: ...... karena ......
- LeNet-5 hanya cocok sebagai baseline atau untuk tugas sederhana seperti MNIST.

## 10. Analisis dan Pembahasan
> **Catatan pengisian:** Isi setiap titik-titik dengan angka dari Bagian 6–9, lalu sesuaikan kalimatnya bila hasilmu berbeda dari dugaan. Penjelasan mekanisme di bawah berasal dari materi Pertemuan 5. Yang harus kamu buktikan adalah apakah angka eksperimenmu mendukungnya.

### 10.1 Kapasitas model: LeNet-5 → AlexNet
LeNet-5 hanya punya ±62 ribu parameter dan 2 layer konvolusi dengan 6/16 filter. Akurasi ujinya ......% dengan gap train-test ...... poin. Jika train acc juga rendah, LeNet-5 **underfitting**: kapasitasnya tidak cukup untuk citra berwarna 10 kelas yang jauh lebih bervariasi daripada MNIST (98,6% di notebook 02).
AlexNet (±28,7 juta parameter, ReLU, Dropout) mencapai ......%. Kenaikan ...... poin ini berasal dari kapasitas yang jauh lebih besar dan ReLU yang tidak mengalami saturasi seperti Tanh (slide 9).

### 10.2 Kedalaman dan filter 3×3: AlexNet → VGG-16
VGG-16 menumpuk 13 conv 3×3. Dua 3×3 bertumpuk memberi receptive field 5×5 dengan bobot 18C² dibanding 25C² (slide 12), plus non-linearitas tambahan. Hasil: VGG-16 ......% vs AlexNet ......%.
Perhatikan distribusi parameter (Bagian 7.3): AlexNet menyimpan ......% parameternya di FC, sedangkan VGG-16 ......%. Pada versi CIFAR, FC6 VGG hanya menerima 512 input (bukan 25.088 seperti versi ImageNet), sehingga porsi FC jauh lebih kecil dari 89% di slide 14.

### 10.3 Lebar dan efisiensi: VGG-16 → GoogLeNet
GoogLeNet memakai ±5,9 juta parameter, ±......× lebih sedikit dari VGG-16, berkat bottleneck 1×1 dan Global Average Pooling yang menggantikan FC raksasa (slide 16–17). Akurasinya ......%.
Namun komputasinya (±1.450 juta MAC) justru yang **terbesar**, karena modul Inception pertama bekerja di resolusi penuh 32×32. Ini terlihat juga dari waktu per epoch (......s). Pelajarannya: **parameter sedikit tidak sama dengan komputasi ringan**.

### 10.4 Residual connection: ResNet-18
ResNet-18 mencapai ......% dengan ±11,2 juta parameter. Shortcut membuat gradien selalu punya jalur langsung (suku +1 pada slide 21), sehingga jaringan 18 layer berbobot dengan BatchNorm dapat dilatih stabil dengan Adam lr 0,001 sejak epoch pertama. Bandingkan kecepatan konvergensinya di epoch 1–3 dengan model lain pada kurva Bagian 7.1.

### 10.5 Efisiensi untuk perangkat mobile: MobileNetV2
MobileNetV2 hanya memakai ±2,2 juta parameter dan ±88 juta MAC (±......× lebih ringan dari GoogLeNet), tetapi mencapai ......%. Depthwise separable convolution memangkas biaya konvolusi ±8–9× (slide 28–29). Selisih akurasinya dengan model terbaik adalah ...... poin. Trade-off ini sepadan untuk aplikasi di ponsel.

### 10.6 Overfitting dan keterbatasan 10 epoch
Model dengan gap terbesar adalah ...... (...... poin). Tanpa augmentasi data, model berkapasitas besar cenderung menghafal data latih. Model dengan akurasi uji yang masih naik di epoch 8–10 adalah ......; model ini kemungkinan masih bisa membaik dengan lebih banyak epoch.

### 10.7 Kelas yang sulit
Kelas tersulit adalah ...... dan ...... (Bagian 8). Kesalahan terbanyak model terbaik adalah ...... diprediksi sebagai ....... Hal ini masuk akal karena ...... (misalnya: kucing dan anjing punya bentuk dan tekstur serupa pada resolusi 32×32).

### 10.8 Keterbatasan eksperimen
1. **Satu seed (42) saja.** Selisih akurasi di bawah ±1 poin belum tentu signifikan; dibutuhkan beberapa seed untuk memastikannya. Operasi GPU juga tidak 100% deterministik, sehingga hasil run ulang bisa berbeda sedikit.
2. **10 epoch dengan learning rate tetap.** Hasil ini mengukur kecepatan belajar dalam anggaran pendek, bukan akurasi maksimum tiap arsitektur.
3. **Adaptasi 32×32.** Model yang dirancang untuk ImageNet telah diubah bagian stem-nya. VGG-16 dan GoogLeNet memakai BatchNorm, yang tidak ada di paper aslinya.
4. **Data uji dipantau setiap epoch** karena pembagian default tidak menyediakan data validasi. Yang dilaporkan sebagai metrik resmi adalah akurasi epoch ke-10.

## 11. Kesimpulan
1. Arsitektur dengan akurasi uji tertinggi adalah ...... (......%), sedangkan yang terendah adalah ...... (......%).
2. Akurasi tidak sebanding lurus dengan jumlah parameter: ......
3. Untuk aplikasi di perangkat terbatas, model yang paling tepat adalah ...... karena ......
4. Untuk akurasi maksimum di server, model yang paling tepat adalah ...... karena ......

---
### Cara mencetak notebook ini ke PDF
1. Pastikan semua sel sudah dijalankan (**Runtime → Run all**) dan semua output terlihat.
2. Isi identitas di bagian atas dan titik-titik di Bagian 10–11.
3. **File → Print** (Ctrl+P) → Destination: **Save as PDF** → Layout: Portrait → centang *Background graphics* → Save.
4. Alternatif (Jupyter lokal): `jupyter nbconvert --to webpdf nama_notebook.ipynb`.